# 03 · Preprocesamiento

Imputación limitada, winsorización y escalado robusto. El preprocesador se guarda para reutilizar exactamente las mismas transformaciones.

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODELS_DIR = ROOT / 'outputs' / 'models'
FIGURES_DIR = ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

modelo = pd.read_parquet(MODELS_DIR / 'dataset_modelo.parquet')
configuracion = json.loads((MODELS_DIR / 'configuracion_variables.json').read_text(encoding='utf-8'))
VARIABLES = configuracion['variables']
IDENTIFICADORES = configuracion['identificadores']
X_original = modelo[VARIABLES].copy()
print(X_original.shape)

## Nulos

Solo se imputan ausencias residuales dentro del universo seleccionado. Si una variable tiene demasiados nulos, vuelve al notebook 02 y retírala.

In [ ]:
resumen_nulos = pd.DataFrame({
    'nulos': X_original.isna().sum(),
    'porcentaje_nulos': X_original.isna().mean().mul(100).round(2),
}).sort_values('porcentaje_nulos', ascending=False)
display(resumen_nulos)

MAXIMO_NULOS_IMPUTABLE = 15.0
problematicas = resumen_nulos.query('porcentaje_nulos > @MAXIMO_NULOS_IMPUTABLE')
if not problematicas.empty:
    raise ValueError(
        'Hay variables con demasiados nulos para una imputación simple: '
        + ', '.join(problematicas.index)
    )

## Winsorización

Se recortan los extremos al 1 % y 99 %. Modifica estos límites y compara la estabilidad de los clústeres.

In [ ]:
CUANTIL_INFERIOR = 0.01
CUANTIL_SUPERIOR = 0.99
limites_inferiores = X_original.quantile(CUANTIL_INFERIOR)
limites_superiores = X_original.quantile(CUANTIL_SUPERIOR)
X_recortado = X_original.clip(limites_inferiores, limites_superiores, axis='columns')

valores_recortados = ((X_original.lt(limites_inferiores)) | (X_original.gt(limites_superiores))).sum()
display(valores_recortados.sort_values(ascending=False).to_frame('valores_recortados'))

## Imputación y escalado

In [ ]:
imputador = SimpleImputer(strategy='median')
escalador = RobustScaler()

X_imputado = imputador.fit_transform(X_recortado)
X = escalador.fit_transform(X_imputado)

if not np.isfinite(X).all():
    raise ValueError('La matriz transformada contiene infinitos o NaN')

print(f'Matriz final: {X.shape}')
display(pd.DataFrame(X, columns=VARIABLES).describe().T)

## Exploración mediante PCA

In [ ]:
pca_completa = PCA().fit(X)
varianza_acumulada = np.cumsum(pca_completa.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada, marker='o')
ax.axhline(0.80, color='tab:red', linestyle='--', label='80 %')
ax.set(xlabel='Número de componentes', ylabel='Varianza explicada acumulada')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'pca_varianza_explicada.png', dpi=160)
plt.show()

## Guardar artefactos

In [ ]:
artefacto = {
    'variables': VARIABLES,
    'cuantil_inferior': CUANTIL_INFERIOR,
    'cuantil_superior': CUANTIL_SUPERIOR,
    'limites_inferiores': limites_inferiores,
    'limites_superiores': limites_superiores,
    'imputador': imputador,
    'escalador': escalador,
}
joblib.dump(artefacto, MODELS_DIR / 'preprocesador.joblib')
np.save(MODELS_DIR / 'matriz_preprocesada.npy', X)
modelo[[*IDENTIFICADORES, 'poblacion_2025']].to_parquet(
    MODELS_DIR / 'municipios_modelo.parquet', index=False
)
print('Artefactos de preprocesamiento guardados')